# TensorBoard-Plots der Big Random Search: gemeinsame Configs

Dieses Notebook wertet den direkten Vergleich zwischen `Pretrained` und `Fully-Joint` aus.
Es werden nur Runs verwendet, deren vergleichbare Hyperparameter-Konfiguration in beiden Varianten exakt vorkommt.

- `no_aux` als **Pretrained**
- `additive` als **Fully-Joint**

Variantenspezifische Unterschiede wie Checkpoint-Initialisierung, Auxiliary-Loss und maximale Trainingsdauer sind nicht Teil des Vergleichsschluessels. Die Kurvenplots werden nicht gekuerzt, damit unterschiedliche Trainingsdauern sichtbar bleiben.


In [ ]:
from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# Thesis-Style: native Breite fuer Einbindung mit width=\textwidth
FIG_FULL_PAIR = (6.0, 2.85)
FIG_FULL_EXPERTS = (6.0, 4.25)

mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})


In [ ]:
RUN_DIR_NAME = "big_randomsearch"
OUT_DIR_NAME = "figures_big_randomsearch"
MAX_EPOCH = None

VARIANTEN = [
    ("no_aux", "Pretrained"),
    ("additive", "Fully-Joint"),
]
MATCH_VARIANTS = [label for _, label in VARIANTEN]

CONFIG_KEY_COLUMNS = [
    "trial",
    "seed",
    "expert_lr",
    "rejector_lr",
    "weight_decay",
    "budget_loss_weight",
    "routing_loss_weight",
    "temperature",
    "topk_noise_std",
    "topk_noise_start_epoch",
    "topk_noise_epochs",
    "batch_size",
    "threshold_r1",
    "threshold_r2",
]


def finde_moe_root() -> Path:
    # Findet den moe-Ordner, egal ob das Notebook aus dem Repo-Root oder aus moe/plots gestartet wird.
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        kandidaten = (
            base,
            base / "moe",
            base / "single_pulse_classifier_training" / "moe",
            base / "WP2-1" / "single_pulse_classifier_training" / "moe",
        )
        for kandidat in kandidaten:
            if (kandidat / "moe_runs" / RUN_DIR_NAME).exists():
                return kandidat
    raise FileNotFoundError(f"moe_runs/{RUN_DIR_NAME} konnte vom aktuellen Arbeitsverzeichnis aus nicht gefunden werden.")


MOE_ROOT = finde_moe_root()
RUN_ROOT = MOE_ROOT / "moe_runs" / RUN_DIR_NAME
OUT_DIR = MOE_ROOT / "plots" / OUT_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ROOT, OUT_DIR


In [ ]:
def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        warnings.warn(f"Konnte JSON nicht lesen: {path}")
        return {}


def dekodiere_float_token(token: str) -> str:
    return token.replace("p", ".").replace("em", "e-")


def parse_run_name(run_name: str) -> dict:
    muster = {
        "trial": r"trial(\d+)",
        "seed": r"seed(\d+)",
        "expert_lr": r"expertlr([0-9pem]+)",
        "rejector_lr": r"rejectorlr([0-9pem]+)",
        "weight_decay": r"wd([0-9pem]+)",
        "temperature": r"temp([0-9pem]+)",
        "topk_noise_std": r"topknoisestd([0-9pem]+)",
        "topk_noise_start_epoch": r"noisestart(\d+)",
        "topk_noise_epochs": r"noiseep(\d+)",
        "budget": r"budget([0-9pem]+)",
        "routing": r"routing([0-9pem]+)",
        "aux": r"aux([0-9pem]+)",
        "aux_warmup": r"auxwarmup(\d+)",
    }
    daten = {}
    for name, pattern in muster.items():
        match = re.search(pattern, run_name)
        if not match:
            daten[name] = np.nan
            continue
        value = match.group(1)
        if name in {"trial", "seed", "aux_warmup", "topk_noise_start_epoch", "topk_noise_epochs"}:
            daten[name] = int(value)
        else:
            daten[name] = float(dekodiere_float_token(value))
    return daten


def nested_get(daten: dict, keys: tuple[str, ...], default=np.nan):
    current = daten
    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


def canonical_number(value):
    if value is None:
        return np.nan
    if isinstance(value, (np.floating, float)):
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    return value


def normalize_config_value(value):
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    if isinstance(value, (np.floating, float)):
        return format(float(value), ".17g")
    if isinstance(value, (np.integer, int)):
        return int(value)
    return value


def make_config_key(row: pd.Series) -> str:
    return json.dumps(
        {column: normalize_config_value(row[column]) for column in CONFIG_KEY_COLUMNS},
        sort_keys=True,
        separators=(",", ":"),
    )


def kurzer_run_name(run: str) -> str:
    parsed = parse_run_name(run)
    teile = []
    if not math.isnan(parsed.get("trial", np.nan)):
        teile.append(f"t{int(parsed['trial'])}")
    if not math.isnan(parsed.get("seed", np.nan)):
        teile.append(f"s{int(parsed['seed'])}")
    if not math.isnan(parsed.get("routing", np.nan)):
        teile.append(f"r{parsed['routing']:g}")
    if not math.isnan(parsed.get("budget", np.nan)):
        teile.append(f"b{parsed['budget']:g}")
    return "_".join(teile) if teile else run[:45]


def safe_name(text: str) -> str:
    return text.lower().replace("-", "_").replace(" ", "_")


def speichere_abbildung(fig, name: str):
    path = OUT_DIR / name
    fig.savefig(path)
    print(path)


In [ ]:
def bool_eq(value, expected: bool) -> bool:
    if value is None:
        return False
    if isinstance(value, float) and np.isnan(value):
        return False
    return bool(value) is expected


def is_canonical_variant_run(row: pd.Series) -> bool:
    if row["variant"] == "Pretrained":
        return (
            row["expert_aux_loss_epochs"] == 0
            and bool_eq(row["only_aux_warmup"], True)
            and row["budget_loss_weight"] == 3.0
            and row["routing_loss_weight"] == 2.0
        )
    if row["variant"] == "Fully-Joint":
        return (
            row["expert_aux_loss_weight"] == 1.0
            and row["expert_aux_loss_epochs"] == 100
            and bool_eq(row["only_aux_warmup"], False)
            and row["aux_loss_mode"] == "additive"
            and row["budget_loss_weight"] == 3.0
            and row["routing_loss_weight"] == 2.0
        )
    return False


def load_tensorboard_scalars(run_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    scalar_rows = []
    run_rows = []

    for folder, label in VARIANTEN:
        variant_root = run_root / folder
        if not variant_root.exists():
            warnings.warn(f"Variante fehlt: {variant_root}")
            continue

        for tensorboard_dir in sorted(variant_root.glob("*/tensorboard")):
            run_dir = tensorboard_dir.parent
            event_files = sorted(tensorboard_dir.glob("events.out.tfevents.*"))
            if not event_files:
                continue

            try:
                acc = EventAccumulator(str(tensorboard_dir), size_guidance={"scalars": 0})
                acc.Reload()
            except Exception as exc:
                warnings.warn(f"TensorBoard-Run konnte nicht gelesen werden: {tensorboard_dir} ({exc})")
                continue

            tags = set(acc.Tags().get("scalars", []))
            parsed = parse_run_name(run_dir.name)
            cfg = read_json(run_dir / "sampled_config.json")
            training = cfg.get("training", {})
            model = cfg.get("model", {})
            loader = cfg.get("loader", {})
            inference = cfg.get("inference", {})
            run_id = f"{folder}/{run_dir.name}"
            n_points = 0

            for tag in sorted(tags):
                for event in acc.Scalars(tag):
                    scalar_rows.append({
                        "variant_folder": folder,
                        "variant": label,
                        "run_id": run_id,
                        "run": run_dir.name,
                        "run_dir": str(run_dir),
                        "tag": tag,
                        "step": event.step,
                        "value": event.value,
                        "wall_time": event.wall_time,
                    })
                    n_points += 1

            run_rows.append({
                "variant_folder": folder,
                "variant": label,
                "run_id": run_id,
                "run": run_dir.name,
                "run_dir": str(run_dir),
                "short_name": kurzer_run_name(run_dir.name),
                "n_points": n_points,
                "trial": canonical_number(parsed.get("trial")),
                "seed": canonical_number(cfg.get("seed", parsed.get("seed"))),
                "expert_lr": canonical_number(training.get("expert_learning_rate", parsed.get("expert_lr"))),
                "rejector_lr": canonical_number(training.get("rejector_learning_rate", parsed.get("rejector_lr"))),
                "weight_decay": canonical_number(training.get("weight_decay", parsed.get("weight_decay"))),
                "budget_loss_weight": canonical_number(training.get("budget_loss_weight", parsed.get("budget"))),
                "routing_loss_weight": canonical_number(training.get("routing_loss_weight", parsed.get("routing"))),
                "temperature": canonical_number(model.get("temperature", parsed.get("temperature"))),
                "topk_noise_std": canonical_number(training.get("topk_noise_std", parsed.get("topk_noise_std"))),
                "topk_noise_start_epoch": canonical_number(training.get("topk_noise_start_epoch", parsed.get("topk_noise_start_epoch"))),
                "topk_noise_epochs": canonical_number(training.get("topk_noise_epochs", parsed.get("topk_noise_epochs"))),
                "batch_size": canonical_number(loader.get("batch_size")),
                "threshold_r1": canonical_number(inference.get("threshold_r1")),
                "threshold_r2": canonical_number(inference.get("threshold_r2")),
                "expert_aux_loss_weight": canonical_number(training.get("expert_aux_loss_weight", parsed.get("aux"))),
                "expert_aux_loss_epochs": canonical_number(training.get("expert_aux_loss_epochs")),
                "aux_loss_mode": training.get("aux_loss_mode"),
                "only_aux_warmup": training.get("only_aux_warmup"),
                "epochs": canonical_number(training.get("epochs")),
            })

    scalars = pd.DataFrame(scalar_rows)
    runs = pd.DataFrame(run_rows)
    if runs.empty:
        return scalars, runs

    runs["config_key"] = runs.apply(make_config_key, axis=1)
    runs["canonical_variant_run"] = runs.apply(is_canonical_variant_run, axis=1)

    if not scalars.empty:
        val_topk = scalars.loc[scalars["tag"] == "val_topk/accuracy"]
        best_topk_full = val_topk.groupby("run_id")["value"].max()
        runs["best_val_topk_accuracy_full"] = runs["run_id"].map(best_topk_full)
        if MAX_EPOCH is not None:
            best_topk_window = val_topk.loc[val_topk["step"] <= MAX_EPOCH].groupby("run_id")["value"].max()
            runs[f"best_val_topk_accuracy_upto_{MAX_EPOCH}"] = runs["run_id"].map(best_topk_window)
        runs["best_val_topk_accuracy"] = runs["best_val_topk_accuracy_full"]

    return scalars, runs


def select_shared_config_runs(scalars_all: pd.DataFrame, runs_all: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    canonical_runs = runs_all.loc[runs_all["canonical_variant_run"]].copy()

    duplicate_mask = canonical_runs.duplicated(["variant", "config_key"], keep=False)
    duplicate_runs = canonical_runs.loc[duplicate_mask].sort_values(["variant", "config_key", "run"])

    sort_columns = ["variant", "config_key", "n_points", "best_val_topk_accuracy", "run"]
    canonical_runs = canonical_runs.sort_values(sort_columns, ascending=[True, True, False, False, True])
    canonical_runs = canonical_runs.drop_duplicates(["variant", "config_key"], keep="first")

    counts = canonical_runs.groupby("config_key")["variant"].nunique()
    shared_keys = set(counts.loc[counts == len(MATCH_VARIANTS)].index)
    duplicate_runs = duplicate_runs.loc[duplicate_runs["config_key"].isin(shared_keys)]
    matched_runs = canonical_runs.loc[canonical_runs["config_key"].isin(shared_keys)].copy()

    fj_best = (
        matched_runs.loc[matched_runs["variant"] == "Fully-Joint", ["config_key", "best_val_topk_accuracy", "seed", "trial"]]
        .rename(columns={"best_val_topk_accuracy": "fully_joint_best"})
    )
    pre_best = (
        matched_runs.loc[matched_runs["variant"] == "Pretrained", ["config_key", "best_val_topk_accuracy"]]
        .rename(columns={"best_val_topk_accuracy": "pretrained_best"})
    )
    order_frame = fj_best.merge(pre_best, on="config_key", how="inner")
    order_frame = order_frame.sort_values(["fully_joint_best", "pretrained_best", "seed", "trial"], ascending=[False, False, True, True])
    config_order = order_frame["config_key"].tolist()
    config_ids = {key: f"C{i:02d}" for i, key in enumerate(config_order, start=1)}

    matched_runs["config_id"] = matched_runs["config_key"].map(config_ids)
    matched_runs["config_label"] = matched_runs.apply(
        lambda row: f"{row['config_id']} | s{int(row['seed'])} | lrE={row['expert_lr']:.2e} | lrR={row['rejector_lr']:.2e} | wd={row['weight_decay']:.0e}",
        axis=1,
    )
    matched_runs["plot_label"] = matched_runs.apply(lambda row: f"Seed: {int(row['seed'])}", axis=1)

    shared_configs = (
        matched_runs[["config_key", "config_id", "config_label", "plot_label", *CONFIG_KEY_COLUMNS]]
        .drop_duplicates("config_key")
        .sort_values("config_id")
    )

    scalars = scalars_all.merge(
        matched_runs[["run_id", "config_key", "config_id", "config_label", "plot_label"]],
        on="run_id",
        how="inner",
    )
    return scalars, matched_runs, shared_configs, duplicate_runs


scalars_all, runs_all = load_tensorboard_scalars(RUN_ROOT)
scalars, runs, shared_configs, duplicate_runs = select_shared_config_runs(scalars_all, runs_all)

runs_all.to_csv(OUT_DIR / "big_randomsearch_summary_all.csv", index=False)
runs.to_csv(OUT_DIR / "big_randomsearch_summary_paired14.csv", index=False)
shared_configs.to_csv(OUT_DIR / "big_randomsearch_shared_configs_paired14.csv", index=False)

print(f"Geladene Runs insgesamt: {len(runs_all)}")
print(f"Gemeinsame Configs: {shared_configs['config_key'].nunique()}")
print(f"Verwendete Runs: {len(runs)}")
display(runs.groupby("variant").size())

if not duplicate_runs.empty:
    print("Doppelte kanonische Runs vor der Auswahl:")
    display(duplicate_runs[["variant", "seed", "trial", "expert_lr", "rejector_lr", "weight_decay", "run"]])

display(shared_configs[["config_id", "seed", "trial", "expert_lr", "rejector_lr", "weight_decay", "budget_loss_weight", "routing_loss_weight"]])
display(runs.sort_values(["config_id", "variant"])[["config_id", "variant", "run", "best_val_topk_accuracy_full"]])


In [ ]:
def tag_frame(tag: str, variant: str | None = None) -> pd.DataFrame:
    frame = scalars.loc[scalars["tag"] == tag].copy()
    if MAX_EPOCH is not None:
        frame = frame.loc[frame["step"] <= MAX_EPOCH]
    if variant is not None:
        frame = frame.loc[frame["variant"] == variant]
    return frame


CONFIG_ORDER = shared_configs["config_key"].tolist()
CONFIG_LABELS = shared_configs.set_index("config_key")["plot_label"].to_dict()
CONFIG_COLORS = {
    key: color
    for key, color in zip(
        CONFIG_ORDER,
        plt.cm.tab20(np.linspace(0, 1, max(len(CONFIG_ORDER), 1))),
    )
}


def style_axis(ax, grid_axis="both"):
    ax.grid(
        axis=grid_axis,
        color="0.90",
        linewidth=0.7,
        linestyle="-",
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def markiere_maximum(ax, frame: pd.DataFrame):
    if frame.empty or frame["value"].dropna().empty:
        return

    max_acc = frame["value"].max()
    yaxis_transform = ax.get_yaxis_transform()

    ax.plot(
        [-0.022, 0.0],
        [max_acc, max_acc],
        transform=yaxis_transform,
        color="0.45",
        linewidth=1.0,
        solid_capstyle="butt",
        clip_on=False,
        zorder=5,
    )

    max_label = f"{100 * max_acc:.2f} %".replace(".", ",")

    ax.text(
        -0.028,
        max_acc,
        max_label,
        transform=yaxis_transform,
        ha="right",
        va="center",
        color="0.35",
        fontsize=7,
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.80,
            "pad": 0.5,
        },
        clip_on=False,
    )


def plot_runs(ax, variant: str, tag: str, markiere_max: bool = False):
    frame = tag_frame(tag, variant)

    for config_key in CONFIG_ORDER:
        run_frame = (
            frame.loc[frame["config_key"] == config_key]
            .sort_values("step")
        )

        if run_frame.empty:
            continue

        ax.plot(
            run_frame["step"],
            run_frame["value"],
            linewidth=1.0,
            alpha=0.90,
            color=CONFIG_COLORS[config_key],
            label=CONFIG_LABELS[config_key],
        )

    if markiere_max:
        markiere_maximum(ax, frame)


def add_bottom_config_legend(fig, ax, y=0.015, ncol=7):
    handles, labels = ax.get_legend_handles_labels()

    if not handles:
        return

    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        ncol=ncol,
        frameon=False,
        fontsize=7,
        columnspacing=0.8,
        labelspacing=0.25,
        handlelength=1.4,
        handletextpad=0.4,
        borderaxespad=0.0,
    )


def set_epoch_xlim(axes, tags):
    if isinstance(tags, str):
        tags = [tags]

    max_step = scalars.loc[
        scalars["tag"].isin(tags),
        "step",
    ].max()

    right = MAX_EPOCH if MAX_EPOCH is not None else max_step

    if pd.notna(right):
        for ax in np.ravel(axes):
            ax.set_xlim(0, right)
            ax.xaxis.set_major_locator(
                mpl.ticker.MaxNLocator(integer=True, nbins=6)
            )


def plot_metric_variant_pair(
    tag: str,
    ylabel: str,
    filename: str,
    y_limits=None,
    markiere_max: bool = False,
):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=FIG_FULL_PAIR,
        sharex=True,
        sharey=True,
        constrained_layout=False,
    )

    for ax, variant in zip(axes, MATCH_VARIANTS):
        plot_runs(
            ax,
            variant,
            tag,
            markiere_max=markiere_max,
        )

        ax.set_title(
            variant,
            pad=3,
        )

        ax.set_xlabel(
            "Epoche",
            labelpad=2,
        )

        ax.yaxis.set_major_formatter(
            mpl.ticker.PercentFormatter(
                xmax=1.0,
                decimals=0,
            )
        )

        if y_limits is not None:
            ax.set_ylim(*y_limits)

        style_axis(
            ax,
            grid_axis="both",
        )

    axes[0].set_ylabel(
        ylabel,
        labelpad=2,
    )

    axes[1].tick_params(
        axis="y",
        labelleft=False,
    )

    set_epoch_xlim(
        axes,
        tag,
    )

    add_bottom_config_legend(
        fig,
        axes[0],
        y=0.015,
        ncol=7,
    )

    fig.subplots_adjust(
        left=0.105,
        right=0.995,
        bottom=0.28,
        top=0.93,
        wspace=0.18,
    )

    speichere_abbildung(
        fig,
        filename,
    )

    plt.show()


def plot_expert_accuracy_variant_pair():
    tags = [
        ("val_expert/small_accuracy", r"$f_{\mathrm{small}}$"),
        ("val_expert/mid_accuracy", r"$f_{\mathrm{mid}}$"),
        ("val_expert/large_accuracy", r"$f_{\mathrm{large}}$"),
    ]

    fig, axes = plt.subplots(
        2,
        3,
        figsize=FIG_FULL_EXPERTS,
        sharex=True,
        sharey=True,
        constrained_layout=False,
    )

    for row, variant in enumerate(MATCH_VARIANTS):
        for col, (tag, expert_label) in enumerate(tags):
            ax = axes[row, col]

            plot_runs(
                ax,
                variant,
                tag,
                markiere_max=False,
            )

            if row == 0:
                ax.set_title(
                    expert_label,
                    pad=3,
                )

            if row == 1:
                ax.set_xlabel(
                    "Epoche",
                    labelpad=2,
                )

            ax.yaxis.set_major_formatter(
                mpl.ticker.PercentFormatter(
                    xmax=1.0,
                    decimals=0,
                )
            )

            style_axis(
                ax,
                grid_axis="both",
            )

        axes[row, 0].set_ylabel(
            f"{variant}\nValidierungsgenauigkeit",
            labelpad=2,
        )

    set_epoch_xlim(
        axes,
        [tag for tag, _ in tags],
    )

    add_bottom_config_legend(
        fig,
        axes[0, 0],
        y=0.01,
        ncol=7,
    )

    fig.subplots_adjust(
        left=0.115,
        right=0.995,
        bottom=0.17,
        top=0.95,
        wspace=0.18,
        hspace=0.26,
    )

    speichere_abbildung(
        fig,
        "big_randomsearch_paired14_val_expert_accuracy.pdf",
    )

    plt.show()


## Top-k-Validierungsgenauigkeit


In [ ]:
plot_metric_variant_pair(
    "val_topk/accuracy",
    "Validierungsgenauigkeit",
    "big_randomsearch_paired14_val_topk_accuracy.pdf",
    markiere_max=True,
)


## Budgetierte Validierungsobergrenze


In [ ]:
plot_metric_variant_pair(
    "val_budgeted_upper_bound",
    "Budgetierte Validierungsobergrenze",
    "big_randomsearch_paired14_val_budgeted_upper_bound.pdf",
    markiere_max=True,
)


## Validierungsgenauigkeit der einzelnen Klassifikatoren


In [ ]:
plot_expert_accuracy_variant_pair()
